# ComfyUI + LTX 2.3 GGUF - Multi-Account Video Generator
Base on: pogscafe (2202 votes) + Lightricks/ComfyUI-LTXVideo (3.9k stars)
---


In [ ]:
BACKEND_NAME = "kaggle-a"
GIST_ID = "8da27f2e6e0d8809a043712cd90f9237"
print(f"Config: {BACKEND_NAME}")

---
## 1. Install Environment

In [ ]:
%%time
import os, sys, subprocess, threading, time, requests, json
from datetime import datetime

home = '/kaggle/working'
COMFY = f'{home}/ComfyUI'
os.chdir(home)

# Use system pip directly - no venv needed on Kaggle
import pip
PIP = lambda args: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + args, check=False)
print('OK')

In [ ]:
%%time
# Clone ComfyUI
if not os.path.exists(COMFY):
    subprocess.run(['git', 'clone', 'https://github.com/comfyanonymous/ComfyUI.git', COMFY], check=False)
os.chdir(COMFY)

# Install minimal deps - Kaggle already has torch, etc
PIP(['safetensors', 'einops', 'sentencepiece', 'requests', 'pyyaml', 'Pillow', 'scipy', 'tqdm', 'psutil'])
# Install ComfyUI native packages
PIP(['comfyui-frontend-package', 'comfyui-workflow-templates', 'comfyui-embedded-docs', 'comfy-kitchen', 'comfy-aimdo', 'av'])
print('ComfyUI ready')

In [ ]:
%%time
# Clone custom nodes
nodes_dir = f'{COMFY}/custom_nodes'
os.chdir(nodes_dir)
for git_url, folder in [
    ('https://github.com/Lightricks/ComfyUI-LTXVideo.git', 'ComfyUI-LTXVideo'),
    ('https://github.com/logtd/ComfyUI-LTXTricks.git', 'ComfyUI-LTXTricks'),
    ('https://github.com/city96/ComfyUI-GGUF.git', 'ComfyUI-GGUF'),
]:
    if not os.path.exists(f'{nodes_dir}/{folder}'):
        subprocess.run(['git', 'clone', git_url, f'{nodes_dir}/{folder}'], check=False)
        print(f'  {folder} OK')
print('All nodes installed')

---
## 2. Download Models

In [ ]:
%%time
# Download LTX-2.3 GGUF Q2_K (12.4 GB) - ONLY format that fits T4 16GB
model_dir = f'{COMFY}/models/unet'
os.makedirs(model_dir, exist_ok=True)
os.chdir(model_dir)

file = 'LTX-2.3-22B-distilled-1.1-Q2_K.gguf'
if not os.path.exists(file):
    url = 'https://huggingface.co/QuantStack/LTX-2.3-GGUF/resolve/main/LTX-2.3-distilled-1.1/LTX-2.3-22B-distilled-1.1-Q2_K.gguf'
    subprocess.run(['wget', '-c', url, '-O', file], check=False)
    sz = os.path.getsize(file) / 1e9
    print(f'Model: {sz:.1f} GB')
else:
    print(f'Model exists: {os.path.getsize(file)/1e9:.1f} GB')

In [ ]:
%%time
# Download text encoder + VAE
for dir_name, dest, src in [
    (f'{COMFY}/models/clip', 'gemma-2b.safetensors', 'text_encoder/model.safetensors'),
    (f'{COMFY}/models/vae', 'ltx-vae.safetensors', 'vae/vae.safetensors'),
]:
    os.makedirs(dir_name, exist_ok=True)
    path = f'{dir_name}/{dest}'
    if not os.path.exists(path):
        url = f'https://huggingface.co/Lightricks/LTX-2/resolve/main/{src}'
        subprocess.run(['wget', '-c', url, '-O', path], check=False)
        print(f'  {dest} OK')
print('Done')

---
## 3. Start ComfyUI + Tunnel

In [ ]:
# Start ComfyUI headless
os.chdir(COMFY)
subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
time.sleep(2)

log = open(f'{home}/comfyui.log', 'w')
proc = subprocess.Popen([sys.executable, 'main.py', '--headless', '--port', '8188',
    '--listen', '127.0.0.1', '--dont-print-server', '--highvram'],
    stdout=log, stderr=log)

for i in range(40):
    time.sleep(3)
    try:
        r = requests.get('http://127.0.0.1:8188/object_info', timeout=2)
        if r.status_code == 200:
            print(f'ComfyUI ready! PID: {proc.pid}')
            break
    except:
        pass
else:
    print('ERROR: ComfyUI did not start')

In [ ]:
# Pinggy tunnel
TUNNEL_URL = None

def pinggy():
    global TUNNEL_URL
    p = subprocess.Popen(['ssh', '-p', '443', '-R0:localhost:8188', 'a.pinggy.io'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    for line in p.stdout:
        if 'https://' in line:
            i = line.find('https://')
            url = line[i:].strip().split()[0]
            TUNNEL_URL = url
            open(f'{home}/tunnel_url.txt', 'w').write(url)
            print(f'TUNNEL_URL: {url}')
            break

t = threading.Thread(target=pinggy, daemon=True)
t.start()
time.sleep(15)

for i in range(30):
    if TUNNEL_URL:
        break
    try:
        TUNNEL_URL = open(f'{home}/tunnel_url.txt').read().strip()
    except:
        pass
    time.sleep(5)

if TUNNEL_URL:
    print(f'Tunnel: {TUNNEL_URL}')
else:
    print('No tunnel available')

---
## 4. Push URL to GitHub Gist

In [ ]:
def update_gist(url, status):
    data = {}
    try:
        r = requests.get(f'https://api.github.com/gists/{GIST_ID}', timeout=5)
        if r.status_code == 200:
            c = r.json()['files']['kaggle_backends.json']['content']
            data = json.loads(c) if c.strip() else {}
    except:
        pass
    data[BACKEND_NAME] = {
        'url': url, 'status': status,
        'updated': datetime.now().isoformat(),
        'capabilities': ['t2v', 'i2v', 'v2v'],
        'gpu': 't4'
    }
    r = requests.patch(f'https://api.github.com/gists/{GIST_ID}',
        json={'files': {'kaggle_backends.json': {'content': json.dumps(data, indent=2)}}},
        timeout=10)
    return r.status_code == 200

if TUNNEL_URL:
    ok = update_gist(TUNNEL_URL, 'online')
    print(f'Gist update: {ok}')
else:
    print('No URL to push')

print(f'\nAPI: POST {TUNNEL_URL}/prompt')

---
## 5. Keep Alive

In [ ]:
print(f'Backend: {BACKEND_NAME}, URL: {TUNNEL_URL}')
try:
    while True:
        time.sleep(300)
        if TUNNEL_URL:
            update_gist(TUNNEL_URL, 'online')
except KeyboardInterrupt:
        if TUNNEL_URL:
            update_gist(TUNNEL_URL, 'offline')
        print('Stopped')